# ⚖️ Regulus — AI Governance Standards Lookup

**Describe an AI issue in plain language; Regulus returns the regulatory provisions that apply — across frameworks, each with a source citation and cited cross-framework references.**

Regulus turns a free-text observation — an audit finding, a model-risk issue, a design question — into a **traceable map of the standards that govern it**. It is built for AI governance and model-risk work, where the hard question is rarely *"what text is similar?"* but *"**which requirement applies, how does it map across frameworks, and can you show me the source?**"*

---

### What this notebook shows

A complete, runnable walkthrough of the system, end to end:

1. **Ingest** real regulatory text from official sources into citable *provisions*.
2. **Look up** the provisions that apply to an issue (semantic retrieval).
3. **Build** a regulatory knowledge graph — frameworks, provisions, risks, and crosswalks.
4. **Answer** with cross-framework, **cited** references — the part a plain vector search cannot do.

The notebook is intentionally thin: each step is a one-line call into the `regulus` package (`standards_loader`, `lookup`, `graph`, `graph_lookup`), so the logic stays testable, reusable, and out of the notebook.

### Why it matters

- **Cross-framework by design.** One issue rarely lives in a single framework; Regulus links equivalent requirements across the EU AI Act, NIST, OECD, ISO, and more.
- **Traceable, not generative-by-default.** Every provision is a real, cited unit of an official framework, and cross-framework mappings (crosswalks) are **curated and cited — never invented by a model.** A governance tool that hallucinates regulatory mappings is a liability.
- **Built on a reusable engine.** Regulus is the applied layer over the [Geometric Knowledge Network (GKN)](https://github.com/minw0607/geometric_knowledge_network) — a retrieval + knowledge-graph substrate designed for exactly this class of typed-relation, evidence-path problems.

### How it works

```text
        Issue / observation
                │
                ▼
   ┌──────────────────────────────┐    Regulatory corpus (5 frameworks):
   │  Retrieve applicable          │◄── EU AI Act · NIST AI RMF · NIST AI 600-1
   │  provisions  (RAG)            │    OECD AI Principles · ISO/IEC 42001
   └──────────────────────────────┘
                │
                ▼
   ┌──────────────────────────────┐    cited CROSSWALK edges +
   │  Expand on the knowledge      │    risk (ADDRESSES) edges
   │  graph                        │
   └──────────────────────────────┘
                │
                ▼
   Applicable provisions  +  cross-framework references (with citations)
```

### Standards & regulations covered

| Framework | What it is | Coverage in this build |
|---|---|---|
| **EU AI Act** | EU regulation on AI (2024/1689) | 113 articles — real text |
| **NIST AI RMF 1.0** | US AI risk-management framework | 72 subcategories — real text |
| **NIST AI 600-1** | Generative-AI profile of the RMF | 49 action groups — real text |
| **OECD AI Principles** | Intergovernmental AI principles | 10 items — real text |
| **ISO/IEC 42001:2023** | AI management-system standard | clause structure only (paywalled — reference) |

*Roadmap:* Fed **SR 26-2** (model risk; supersedes SR 11-7) and further frameworks slot into the same ingestion → graph → lookup pipeline.

### How to use it

Run the cells in order. In Steps 3 and 5, replace the issue string with your own observation — for example:

> *"We deployed a credit model without testing for demographic bias."*
> *"We run real-time facial recognition in public spaces for law enforcement."*
> *"There is no post-deployment monitoring for our high-risk AI system."*

**Illustrative result** — for the bias example, Regulus surfaces **NIST AI RMF `MEASURE 2.11`** (fairness & bias) and cross-walks it to **EU AI Act `Article 10`** (data governance) and the **NIST GenAI profile**, each with its source: a starting map of the applicable standards and their equivalents across frameworks.

> **Status:** early-stage MVP. Retrieval and the cited crosswalk graph work end-to-end today; LLM interpretation, full evidence paths, and a web UI are on the roadmap. Always verify results against the authoritative source.

## 1. Setup

`ensure_gkn()` makes the [Geometric Knowledge Network](https://github.com/minw0607/geometric_knowledge_network)
importable — the installed package if present, otherwise the local sibling checkout —
so this notebook runs **without a `pip install`** in any kernel that has the usual
scientific stack (numpy, pandas, scikit-learn, networkx).

The default retriever is **TF-IDF** (no API keys). For higher-quality retrieval,
set `REGULUS_RETRIEVER=embedding` in a `.env` file (it reuses GKN's embedding
store — Azure/OpenAI or a local model); see `.env.example`.

In [ ]:
# Auto-reload edited modules so code changes take effect without restarting the kernel.
try:
    _ip = get_ipython(); _ip.run_line_magic('load_ext', 'autoreload'); _ip.run_line_magic('autoreload', '2')
except Exception:
    pass

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))   # make `regulus` importable

from regulus import demo
demo.ensure_gkn()
config = demo.config()
print('retriever:', config.retriever, '| top_k:', config.top_k)

## 2. Ingest the standards

Regulus fetches **real** regulatory text from official sources and splits it into
citable *provisions*, across five frameworks:

| Framework | Provisions | Source |
|---|---|---|
| **EU AI Act** | 113 articles | EUR-Lex (© EU; reuse with attribution) |
| **NIST AI RMF 1.0** | 72 subcategories | NIST PDF (U.S. Government work) |
| **NIST AI 600-1** (GenAI Profile) | 49 action groups | NIST PDF — keyed to the AI RMF |
| **OECD AI Principles** | 5 principles + 5 recommendations | OECD/LEGAL/0449 |
| **ISO/IEC 42001:2023** | 16 clauses/controls | reference-only (paywalled — titles/structure, no normative text) |

Sources are downloaded and **cached** under `data/standards_cache/` on first run.
Parsing PDFs needs `pypdf`; if it isn't installed, Regulus falls back to
**committed snapshots**, so this step works either way. Each provision keeps its
`source_url` — Regulus never returns an uncited result.

In [ ]:
provisions = demo.load_provisions(config)
demo.provisions_summary(provisions)

## 3. Look up the applicable provisions

Given a free-text issue, Regulus retrieves the provisions whose text is the closest
match. `score` is a relative similarity (higher = closer); with TF-IDF the absolute
values are small — read them as a ranking, not a probability.

This is the **direct hit** — the provision the issue most looks like. The
cross-framework links come in Step 5.

**Try it:** edit the issue string below to your own observation.

In [ ]:
lookup = demo.baseline_lookup(provisions, config)
demo.lookup_table(lookup, 'We run real-time facial recognition in public spaces to assist law enforcement.')

## 4. Build the regulatory knowledge graph

The provisions become a graph:

- `Framework` **contains** `Provision` nodes;
- `Provision` **addresses** `RiskCategory` nodes (the seven NIST trustworthiness characteristics) — these tags are *keyword-derived and low-confidence*, meant for navigation;
- `Provision` ↔ `Provision` **crosswalk** edges link equivalent concerns across frameworks.

**Governance rule:** crosswalk edges come **only** from the curated, cited table
[`data/crosswalks/crosswalks.csv`](../data/crosswalks/crosswalks.csv) — never inferred by a model. To extend the graph, edit that
CSV (add rows with a `source`/citation) or add frameworks; nothing else changes.

In [ ]:
gl = demo.crosswalk_lookup(provisions, config)
demo.graph_stats(gl)

## 5. Look up with cross-framework crosswalks

The payoff: each applicable provision now carries the **risks** it addresses and
its **cited cross-framework references** — the same concern, linked to another
framework, with the mapping's citation. This is what a plain vector search cannot
do.

Read a row as: *"for this issue, this provision applies; it concerns these risks;
and here is the equivalent guidance in another framework (with its source)."*
Cross-framework references appear only when both frameworks are loaded and a
curated crosswalk connects them.

In [ ]:
demo.crosswalk_table(gl, 'Our credit model was deployed without testing for demographic bias.')

## 6. What's next

- **Evidence paths** — use GKN's multi-hop retriever + path explainer to return the full trail (issue → provision → crosswalk → provision).
- **Interpretation** — an LLM layer that turns the retrieved provisions + paths into a structured, cited answer (risks · standards · cross-refs · guidance).
- **Interface + evaluation** — a "submit an issue" UI and a benchmark of issue → expected-standards and crosswalk accuracy.

**Extending Regulus:** add crosswalk rows or frameworks in `data/`, or swap the seed
crosswalks for authoritative mappings. For embedding-quality retrieval, set
`REGULUS_RETRIEVER=embedding` in `.env`.